In [120]:
# !pip install ipython-sql sqlalchemy pandas prettytable

In [121]:
dataset_link = 'https://www.kaggle.com/datasets/testdatabox/finance-fraud-and-loans-dataset-testdatabox'

In [122]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [123]:
import sqlite3
import pandas as pd
from sklearn.impute import SimpleImputer

import prettytable
# Set the default display format for prettytable to 'DEFAULT' (i.e., a simple table format)
prettytable.DEFAULT = 'DEFAULT'

In [124]:
# Load data file
df1  = pd.read_csv("account_status.csv")
df2 = pd.read_csv("account_types.csv")
df3 = pd.read_csv("accounts.csv")
df4 = pd.read_csv("addresses.csv")
df5 = pd.read_csv("branches.csv")
df6 = pd.read_csv("customer_types.csv")
df7 = pd.read_csv("customers.csv")
df8 = pd.read_csv("loan_status.csv")
df9 = pd.read_csv("loans.csv")
df10 = pd.read_csv("transaction_types.csv")
df11 = pd.read_csv("transactions.csv")

In [125]:
# Connect to a sqlite db
conn = sqlite3.connect('fraud_detection.db')

In [126]:
# %sql sqlite:///fraud_detection.db

In [127]:
# Load data file to sqlite
# if exists: fail, replace, append
df1.to_sql("account_status", conn, if_exists = 'replace', index = False)
df2.to_sql("account_types", conn, if_exists = 'replace', index = False)
df3.to_sql("accounts", conn, if_exists = 'replace', index = False)
df4.to_sql("addresses", conn, if_exists = 'replace', index = False)
df5.to_sql("branches", conn, if_exists = 'replace', index = False)
df6.to_sql("customer_types", conn, if_exists = 'replace', index = False)
df7.to_sql("customers", conn, if_exists = 'replace', index = False)
df8.to_sql("loan_status", conn, if_exists = 'replace', index = False)
df9.to_sql("loans", conn, if_exists = 'replace', index = False)
df10.to_sql("transaction_types", conn, if_exists = 'replace', index = False)
df11.to_sql("transactions", conn, if_exists = 'replace', index = False)

50000

In [128]:
# df11 is transactions

print(df7.columns)
# print(df11.sample(3))

Index(['CustomerID', 'FirstName', 'LastName', 'DateOfBirth', 'AddressID',
       'CustomerTypeID'],
      dtype='str')


## Cleaning Data with pandas

### Assessing missing values and imputing

#### Accounts table

In [129]:
df3.isna().sum()

AccountID           0
CustomerID          0
AccountTypeID       0
AccountStatusID     0
Balance             0
OpeningDate        33
dtype: int64

In [130]:
date_imputer = SimpleImputer(strategy='most_frequent')
df3['OpeningDate'] = pd.Series(
    date_imputer.fit_transform(df3[['OpeningDate']]).ravel()
)


In [131]:
df3.OpeningDate.sample()

397    2019-07-17 00:00:00.000000
Name: OpeningDate, dtype: str

In [132]:
# Convert to datetime
df3['OpeningDate'] = pd.to_datetime(df3['OpeningDate'])

# Extract useful date features for SQL analysis
df3['Year'] = df3['OpeningDate'].dt.year
df3['Month'] = df3['OpeningDate'].dt.month
df3['Day'] = df3['OpeningDate'].dt.day
df3['DayOfWeek'] = df3['OpeningDate'].dt.dayofweek

In [133]:
# Filtering out any future dates
df3.loc[df3['OpeningDate'] > pd.Timestamp.now(), 'OpeningDate']

1442   2026-07-06 15:01:42.900415
Name: OpeningDate, dtype: datetime64[us]

In [134]:
df3.loc[df3['OpeningDate'] > pd.Timestamp.now(), 'OpeningDate'] = pd.Timestamp.now()


#### Addresses Table

In [135]:
df4.isna().sum()

AddressID     0
Street       24
City         26
Country      24
dtype: int64

In [136]:
text_imputer = SimpleImputer(strategy='constant', fill_value='Unknown')
df4[['Street', 'City', 'Country']] = text_imputer.fit_transform(df4[['Street', 'City', 'Country']])

In [137]:
df4.isna().sum()

AddressID    0
Street       0
City         0
Country      0
dtype: int64

#### Customers table

In [138]:
df7.isna().sum()

CustomerID         0
FirstName         22
LastName          23
DateOfBirth        0
AddressID          0
CustomerTypeID     0
dtype: int64

In [139]:
text_imputer = SimpleImputer(strategy='constant', fill_value='Unknown')
df7[['FirstName', 'LastName']] = text_imputer.fit_transform(df7[['FirstName', 'LastName']])

In [140]:
df7.isna().sum()

CustomerID        0
FirstName         0
LastName          0
DateOfBirth       0
AddressID         0
CustomerTypeID    0
dtype: int64

In [141]:
# Clean names
df7['FirstName'] = df7['FirstName'].str.strip().str.title()
df7['LastName'] = df7['LastName'].str.strip().str.title()

In [142]:
df7['DateOfBirth'].sample(5)

856    1981-01-27 00:00:00.000000
4      1966-02-20 00:00:00.000000
47     1972-10-02 00:00:00.000000
540    1983-09-19 00:00:00.000000
268    1962-06-10 00:00:00.000000
Name: DateOfBirth, dtype: str

In [143]:
def convert_dates_safe(date_series):
    """Try multiple date formats and handle errors gracefully"""
    formats = [
        '%Y-%m-%d %H:%M:%S.%f',  # 1978-08-01 00:00:00.000000
        '%Y-%m-%dT%H:%M:%S',      # 1960-08-27T00:00:00
        '%Y-%m-%d %H:%M:%S',      # 1978-08-01 00:00:00
        '%Y-%m-%d',               # 1978-08-01
        '%m/%d/%Y',               # 08/01/1978
        '%d/%m/%Y',               # 01/08/1978
    ]
    
    for fmt in formats:
        try:
            return pd.to_datetime(date_series, format=fmt, errors='raise')
        except:
            continue
    
    # If all formats fail, use pandas' flexible parser
    return pd.to_datetime(date_series, errors='coerce')

# Apply the function
df7['DateOfBirth'] = convert_dates_safe(df7['DateOfBirth'])

# Extract features
df7['Year'] = df7['DateOfBirth'].dt.year
df7['Month'] = df7['DateOfBirth'].dt.month
df7['Day'] = df7['DateOfBirth'].dt.day
df7['DayOfWeek'] = df7['DateOfBirth'].dt.dayofweek

#### Loans Table

In [144]:
df9.isna().sum()

LoanID              0
AccountID           0
LoanStatusID        0
PrincipalAmount     0
InterestRate        0
StartDate           6
EstimatedEndDate    6
dtype: int64

In [145]:
date_imputer = SimpleImputer(strategy='most_frequent')
df9[['StartDate', 'EstimatedEndDate']] = date_imputer.fit_transform(df9[['StartDate', 'EstimatedEndDate']])

In [146]:
df9.isna().sum()

LoanID              0
AccountID           0
LoanStatusID        0
PrincipalAmount     0
InterestRate        0
StartDate           0
EstimatedEndDate    0
dtype: int64

In [147]:
# Convert to datetime
df9['StartDate'] = pd.to_datetime(df9['StartDate'])
df9['EstimatedEndDate'] = pd.to_datetime(df9['EstimatedEndDate'])

#### Transactions Table 

In [148]:
df11.isna().sum()

TransactionID              0
AccountOriginID            0
AccountDestinationID       0
TransactionTypeID          0
Amount                     0
TransactionDate         1000
BranchID                   0
Description                0
dtype: int64

In [149]:
df11.sample()

,TransactionID,AccountOriginID,AccountDestinationID,TransactionTypeID,Amount,TransactionDate,BranchID,Description
13130,3042762,201438,200013,3,4572.21,2022-08-01 19:00:00.000000,16,Transaction 42762


In [150]:
df11['TransactionDate'] = pd.to_datetime(df11['TransactionDate'])

In [151]:
# checking if there are any future dates and replacing them with the current date
df11.loc[df11['TransactionDate'] > pd.Timestamp.now(), 'TransactionDate']
df11.loc[df11['TransactionDate'] > pd.Timestamp.now(), 'TransactionDate'] = pd.Timestamp.now()

In [152]:
# date_imputer = SimpleImputer(strategy='most_frequent')
# df11['TransactionDate'] = date_imputer.fit_transform(df11[['TransactionDate']]).ravel()

from sklearn.impute import SimpleImputer
import pandas as pd

# Convert datetime to string first
df11['TransactionDate_str'] = df11['TransactionDate'].astype(str)

# Impute with most frequent date
date_imputer = SimpleImputer(strategy='most_frequent')
df11['TransactionDate_imputed'] = date_imputer.fit_transform(df11[['TransactionDate_str']]).ravel()

# Convert back to datetime
df11['TransactionDate'] = pd.to_datetime(df11['TransactionDate_imputed'])

# Drop temporary columns
df11.drop(['TransactionDate_str', 'TransactionDate_imputed'], axis=1, inplace=True)

In [153]:
df11.isna().sum()

TransactionID           0
AccountOriginID         0
AccountDestinationID    0
TransactionTypeID       0
Amount                  0
TransactionDate         0
BranchID                0
Description             0
dtype: int64

In [154]:
# Convert to datetime
df11['TransactionDate'] = pd.to_datetime(df11['TransactionDate'])

# Extract useful date features for SQL analysis
df11['Year'] = df11['TransactionDate'].dt.year
df11['Month'] = df11['TransactionDate'].dt.month
df11['Day'] = df11['TransactionDate'].dt.day
df11['DayOfWeek'] = df11['TransactionDate'].dt.dayofweek


In [155]:
clean_conn = sqlite3.connect('fraud_detection_cleaned.db')

df1.to_sql("account_status", clean_conn, if_exists='replace', index=False)
df2.to_sql("account_types", clean_conn, if_exists='replace', index=False)
df3.to_sql("accounts", clean_conn, if_exists='replace', index=False)
df4.to_sql("addresses", clean_conn, if_exists='replace', index=False)
df5.to_sql("branches", clean_conn, if_exists='replace', index=False)
df6.to_sql("customer_types", clean_conn, if_exists='replace', index=False)
df7.to_sql("customers", clean_conn, if_exists='replace', index=False)
df8.to_sql("loan_status", clean_conn, if_exists='replace', index=False)
df9.to_sql("loans", clean_conn, if_exists='replace', index=False)
df10.to_sql("transaction_types", clean_conn, if_exists='replace', index=False)
df11.to_sql("transactions", clean_conn, if_exists='replace', index=False)


50000

In [156]:
clean_conn.close()

In [157]:
# Connect ONLY to the cleaned database
%sql sqlite:///fraud_detection_cleaned.db

# Verify only one connection is active
%sql -l

{'sqlite:///fraud_detection_cleaned.db': <sql.connection.Connection at 0x7fb6bbcdb050>}

Get names of all tables from the database

In [158]:
%%sql 
select name
from sqlite_master
where type = 'table'

 * sqlite:///fraud_detection_cleaned.db
Done.


name
account_status
account_types
accounts
addresses
branches
customer_types
customers
loan_status
loans
transaction_types


Get names of all columns from a table

In [159]:
%%sql
pragma table_info("customers")

 * sqlite:///fraud_detection_cleaned.db
Done.


cid,name,type,notnull,dflt_value,pk
0,CustomerID,INTEGER,0,None,0
1,FirstName,TEXT,0,None,0
2,LastName,TEXT,0,None,0
3,DateOfBirth,TIMESTAMP,0,None,0
4,AddressID,INTEGER,0,None,0
5,CustomerTypeID,INTEGER,0,None,0
6,Year,REAL,0,None,0
7,Month,REAL,0,None,0
8,Day,REAL,0,None,0
9,DayOfWeek,REAL,0,None,0


## Analysis with SQL

1. Get information about customers

In [160]:
%%sql
select CustomerID, FirstName, LastName, DateOfBirth, ct.TypeName
from customers c, customer_types ct 
on c.CustomerTypeID = ct.CustomerTypeID
limit 5

 * sqlite:///fraud_detection_cleaned.db
Done.


CustomerID,FirstName,LastName,DateOfBirth,TypeName
10832,Nyla,Aguirre,1974-02-07 00:00:00,Individual
10983,Unknown,Battle,1963-02-01 00:00:00,Small Business
10837,Angelena,Harrington,1964-03-25 00:00:00,Large Enterprise
10107,Remona,Glass,1965-09-16 00:00:00,Individual
10553,King,Becker,1966-02-20 00:00:00,Large Enterprise


2. List all active loans with their details

In [161]:
%%sql
select l.LoanID, ls.StatusName ,l.AccountID, l.PrincipalAmount, l.InterestRate, l.StartDate, l.EstimatedEndDate
from loans l, loan_status ls
where ls.StatusName == 'Active'
order by l.PrincipalAmount desc
limit 5

 * sqlite:///fraud_detection_cleaned.db
Done.


LoanID,StatusName,AccountID,PrincipalAmount,InterestRate,StartDate,EstimatedEndDate
400095,Active,200694,99830.33,0.143,2022-07-24 00:00:00,2027-01-20 00:00:00
400048,Active,201489,99830.19,0.1134,2022-10-31 00:00:00,2024-03-09 00:00:00
400060,Active,200266,99733.08,0.1017,2022-01-18 00:00:00,2024-07-04 00:00:00
400113,Active,200304,99488.79,0.1418,2021-04-05 00:00:00,2024-05-05 00:00:00
400165,Active,201475,99467.25,0.1438,2022-07-27 00:00:00,2025-09-27 00:00:00


3. Transaction Type Distribution

In [162]:
%%sql
select tt.TypeName, count(t.TransactionID) as count, sum(t.Amount) as sum
from transactions t, transaction_types tt
on t.TransactionTypeID = tt.TransactionTypeID
group by tt.TypeName


 * sqlite:///fraud_detection_cleaned.db
Done.


TypeName,count,sum
Deposit,15218,37981288.32
Payment,5014,12533074.71
Transfer,14904,37534177.4
Withdrawal,14864,37146059.07


4. Count customer by types

In [163]:
%%sql
select count(CustomerID) as count, ct.TypeName
from customers c, customer_types ct 
on c.CustomerTypeID = ct.CustomerTypeID
group by ct.TypeName
order by count desc

 * sqlite:///fraud_detection_cleaned.db
Done.


count,TypeName
402,Large Enterprise
356,Small Business
353,Individual


5. List all transactions with amount greater than $3,000


In [164]:
%%sql
select max(Amount), min(Amount)
from transactions

 * sqlite:///fraud_detection_cleaned.db
Done.


max(Amount),min(Amount)
4999.59,1.01


In [165]:
%%sql
select * 
from transactions 
where Amount > 3000
order by Amount desc
limit 5

 * sqlite:///fraud_detection_cleaned.db
Done.


TransactionID,AccountOriginID,AccountDestinationID,TransactionTypeID,Amount,TransactionDate,BranchID,Description,Year,Month,Day,DayOfWeek
3048536,201288,200756,3,4999.59,2021-03-29 22:00:00,13,Transaction 48536,2021,3,29,0
3037709,200786,201566,2,4999.47,2021-02-22 09:00:00,15,Transaction 37709,2021,2,22,0
3048218,201364,201153,3,4999.4,2021-05-30 02:00:00,14,Transaction 48218,2021,5,30,6
3042375,200375,200712,1,4999.37,2024-01-11 08:00:00,11,Transaction 42375,2024,1,11,3
3045037,200830,200180,1,4999.37,2023-04-01 12:00:00,3,Transaction 45037,2023,4,1,5


6. Age distribution of customers

In [171]:
%%sql
select Year, count(Year) as count
from customers
group by Year
order by count(Year) desc

 * sqlite:///fraud_detection_cleaned.db
Done.


Year,count
1969.0,34
1967.0,34
1987.0,33
2000.0,31
1976.0,31
1966.0,31
1961.0,31
1977.0,30
1998.0,29
1993.0,29
